# Data preprocessing

In [1]:
import numpy as np
import uuid
from tqdm.auto import tqdm

In [2]:
!pip install PyMuPDF

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.1/24.1 MB 71.8 MB/s eta 0:00:00:00:0100:01


In [4]:
import fitz  # PyMuPDF

doc = fitz.open("/kaggle/input/casml-dataset/Dataset_RAG (1)/book.pdf")
pages = []
page_start_positions = []  # сохраняем, где начинается каждая страница в общем тексте
text = ""
for i, page in enumerate(doc):
    page_start_positions.append(len(text))
    text += page.get_text("text") + "\n"
    pages.append(page.get_text("text"))



with open("textbook.txt", "w", encoding="utf-8") as f:
    f.write(text)


print(f"Извлечено {len(pages)} страниц. Общая длина текста: {len(text):,} символов.")


Извлечено 753 страниц. Общая длина текста: 2,256,061 символов.


In [5]:
with open("textbook2.txt", "w", encoding="utf-8") as f:
    f.write(text)

In [7]:
# print(text[4522:10300])

# Model

In [ ]:
from transformers import pipeline
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(device)


model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-7B-Instruct",
    torch_dtype="auto",
    device_map="auto"
)

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2-1.5B-Instruct")

prompt = "Give me a short introduction to large language model."
messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": prompt}
]

In [ ]:
generation_pipeline = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    torch_dtype=torch.float16,
)

print(generation_pipeline(messages, max_new_tokens=256, do_sample=True, temperature=0.3, top_p=0.9)[0]['generated_text'][-1]['content'])

# vector bd

In [ ]:
# import torch.nn.functional as F
# from transformers import AutoModel

from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("intfloat/multilingual-e5-large", model_kwargs={'torch_dtype': torch.float16})

In [ ]:
!pip install langchain-qdrant

In [ ]:
from qdrant_client import QdrantClient, models

client = QdrantClient(":memory:")

client.create_collection(
    collection_name="psycology_e5",
    on_disk_payload=True,
    vectors_config=models.VectorParams(
        size=1024,
        distance=models.Distance.COSINE,
        on_disk=True
    ),
)

# chunk-split

In [ ]:
!pip install langchain-community

In [ ]:
def find_page_for_chunk(start_index, page_starts):
    """Находит, на какой странице начинается чанк по его позиции в тексте."""
    if start_index is None or start_index < 0:
        return 1  # fallback — если не нашли позицию, считаем первой страницей
    for i, p_start in enumerate(page_starts):
        if start_index < p_start:
            return max(1, i)  # страницы нумеруются с 1
    return len(page_starts) 

In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1500, chunk_overlap=100, separators=["\n\n", "\n", " ", ""])

In [ ]:
chunks = []
last_pos = 0
for chunk in text_splitter.split_text(text):
    if not chunk.strip():  # пропускаем пустые чанки
        continue

    start_index = text.find(chunk, last_pos)
    if start_index == -1:
        # если find не нашёл (например, повтор текста), пробуем искать с начала
        start_index = text.find(chunk)
        if start_index == -1:
            # если вообще не нашли — пропускаем этот чанк
            print(f"Не найден чанк{chunk} в тексте, пропущен.")
            continue

    page = find_page_for_chunk(start_index, page_start_positions)
    chunks.append({"text": chunk, "page": page})
    last_pos = start_index + len(chunk)

print(f"Получено {len(chunks)} чанков.")

In [ ]:
vectors = embedding_model.encode([chunk["text"] for chunk in chunks],
                                 batch_size=32, device=device, normalize_embeddings=True, show_progress_bar=True).tolist()

In [ ]:
batch_size = 64

for i in tqdm(range(0, len(vectors), batch_size)):
    batch_points = [
        models.PointStruct(
            id=str(uuid.uuid4()),
            vector=vectors[j],
            payload={
                'text': chunks[j]["text"],
                'page': chunks[j]["page"],
            }
        )
        for j in range(i, min(i + batch_size, len(vectors)))
    ]
    client.upsert(collection_name='psycology_e5', points=batch_points)


# Submission

In [53]:
def semantic_search(client, query, limit=10, collection_name="psycology_e5"):
    """
    Выполняет семантический поиск в коллекции Qdrant.

    Аргументы:
        client: экземпляр QdrantClient
        query: текстовый запрос пользователя
        limit: количество возвращаемых чанков (по умолчанию 10)
        collection_name: имя коллекции (по умолчанию 'psycology_e5')

    Возвращает:
        Список словарей формата:
        [
            {
                "text": "...",          # сам чанк
                "page": 123,            # страница, если есть в payload
                "score": 0.87           # косинусная близость
            },
            ...
        ]
    """
    # Кодируем запрос
    query_vector = embedding_model.encode(
        query,
        normalize_embeddings=True,
        device=device
    ).tolist()

    # Делаем запрос в Qdrant
    hits = client.search(
        collection_name=collection_name,
        query_vector=query_vector,
        limit=limit
    )

    # Безопасно обрабатываем результаты
    if not hits:
        print("Предупреждение: по запросу ничего не найдено.")
        return []

    # Собираем полезную информацию из результатов
    results = []
    for hit in hits:
        payload = hit.payload or {}
        results.append({
            "text": payload.get("text", ""),
            "page": payload.get("page", None),
            "score": hit.score
        })

    return results


In [54]:
def llm_answer(query, context):
    """
    Генерирует ответ на вопрос по фрагментам учебника (RAG).
    Использует Qwen2-1.5B-Instruct для точного, достоверного ответа.
    """
    prompt = f"""Текст из учебника психологии:
{context}

Вопрос:
{query}"""

    messages = [
        {
            "role": "system",
            "content": (
                "You are a very skeptical scientist in the field of psychology. You will receive a context consisting of text clippings from a book on the desired topic. Your task is to answer the user as accurately and honestly as possible. Make sure that the answer is detailed, specific, and directly related to the question. Do not add information that is not directly supported by the provided clippings from the book. If there is no direct answer in the text, tell me about it honestly."
            ),
        },
        {"role": "user", "content": prompt},
    ]

    output = generation_pipeline(
        messages,
        max_new_tokens=512,
        do_sample=True,
        temperature=0.2,
        top_p=0.9,
    )

    # Проверяем структуру вывода, т.к. она может отличаться между версиями Transformers
    if isinstance(output[0]["generated_text"], list):
        # новый формат: список сообщений
        return output[0]["generated_text"][-1]["content"]
    elif isinstance(output[0]["generated_text"], str):
        # старый формат: просто строка
        return output[0]["generated_text"]
    else:
        # fallback
        return str(output[0])


In [55]:
import json
import pandas as pd

queries = json.load(open("/kaggle/input/casml-dataset/Dataset_RAG (1)/queries.json"))

results = []
for q in tqdm(queries):
    query = q["question"]
    query_id = q["query_id"]

    relevant_chunks = semantic_search(client, query, limit=5)
    context = "\n\n".join([chunk["text"] for chunk in relevant_chunks])
    pages = sorted({chunk["page"] for chunk in relevant_chunks if chunk.get("page") is not None})
    # pages = sorted(list({chunk["page"] for chunk in relevant_chunks}))
    references = json.dumps({"pages": pages})

    answer = llm_answer(query, context)

    results.append({
        "ID": query_id,
        "context": context,
        "answer": answer,
        "references": references
    })

df = pd.DataFrame(results)
df.to_csv("submission_basic_rag1.csv", index=False)
print("submission_basic_rag1.csv saved")


  0%|          | 0/50 [00:00<?, ?it/s]

/tmp/ipykernel_48/3226289914.py:30: DeprecationWarning: `search` method is deprecated and will be removed in the future. Use `query_points` instead.
  hits = client.search(


submission_basic_rag1.csv saved


# With sections

In [206]:
import re

# --- Шаг 1. Шаблоны ---
chapter_pattern = re.compile(r'CHAPTER\s+(\d+)\s*([A-Za-z\s,:;-]{2,100})', re.IGNORECASE)
section_pattern = re.compile(
    r'(?:(?<=\n)|(?<=\s)|(?<=\.)|(?<=^))(\d{1,2}\.\d{1,2})\s+([A-Z][A-Za-z\s,&:;\-]{3,100})'
)

raw_sections = []
for match in chapter_pattern.finditer(text):
    raw_sections.append((match.start(), f"chapter_{match.group(1)}", match.group(2).strip()))
for match in section_pattern.finditer(text):
    raw_sections.append((match.start(), match.group(1), match.group(2).strip()))

raw_sections.sort(key=lambda x: x[0])

# --- Шаг 2. Определяем конец оглавления ---
text_lower = text.lower()
toc_match = re.search(r'chapter\s+1[^a-z0-9]{0,20}introduction', text_lower)
toc_end = toc_match.end() if toc_match else int(len(text) * 0.05)

# --- Шаг 3. Фильтрация реальных секций ---
section_positions = []

for pos, sec_id, sec_name in raw_sections:
    if pos < toc_end:
        continue

    # Окно вокруг и после заголовка
    snippet_after = text[pos:pos + 2000].lower()
    snippet_before = text[max(0, pos - 500):pos].lower()

    # Пропускаем служебные куски
    context = text_lower[max(0, pos - 300): pos + 300]
    if any(bad in context for bad in ["chapter outline", "table of contents", "appendix", "references", "figure", "table"]):
        continue

    # Проверяем наличие LEARNING OBJECTIVES поблизости
    # Может быть в 2000 символах после или в 500 до
    has_lo_after = re.search(r'learning\s*objectives', snippet_after, re.I)
    has_lo_before = re.search(r'learning\s*objectives', snippet_before, re.I)
    if not (has_lo_after or has_lo_before):
        continue  # если нет — не добавляем

    section_positions.append((pos, sec_id, sec_name.strip()))

section_positions.sort(key=lambda x: x[0])

print(f"Найдено {len(section_positions)} реальных секций/глав (с LEARNING OBJECTIVES поблизости).")
for s in section_positions:
    print(s)


✅ Найдено 81 реальных секций/глав (с LEARNING OBJECTIVES поблизости).
(27871, '1.1', 'What Is Psychology')
(33261, '1.2', 'History of Psychology\nLEARNING OBJECTIVES\nBy the end of this section, you will be able to:')
(61296, '1.3', 'Contemporary Psychology\nLEARNING OBJECTIVES\nBy the end of this section, you will be able to:')
(84290, '1.4', 'Careers in Psychology\nLEARNING OBJECTIVES\nBy the end of this section, you will be able to:')
(107568, '2.1', 'Why Is Research Important')
(127030, '2.2', 'Approaches to Research\nLEARNING OBJECTIVES\nBy the end of this section, you will be able to:')
(151489, '2.3', 'Analyzing Findings\nLEARNING OBJECTIVES\nBy the end of this section, you will be able to:')
(188618, '2.4', 'Ethics\nLEARNING OBJECTIVES\nBy the end of this section, you will be able to:')
(219932, '3.1', 'Human Genetics\nLEARNING OBJECTIVES\nBy the end of this section, you will be able to:')
(241098, '3.2', 'Cells of the Nervous System\nLEARNING OBJECTIVES\nBy the end of this sec

In [207]:
# если меньше 88, добавляем недостающие вручную
missing_manual = [
    (292881, "3.5", "The Endocrine System"),
    (484673, "5.6", "Gestalt Principles of Perception"),
    (647998, "7.4", "What Are Intelligence and Creativity?"),
    (680376, "7.6", "The Source of Intelligence"),
    (776101, "8.4", "Ways to Enhance Memory"),
    (970099, "10.3", "Sexual Behavior"),
    (1079861, "11.4", "Learning Approaches"),
    (1111908, "11.8", "Cultural Understandings of Personality"),
    (1171573, "12.2", "Self-presentation"),
    (1224415, "12.5", "Prejudice and Discrimination"),
]

section_positions.extend(missing_manual)
section_positions.sort(key=lambda x: x[0])

to_remove_ids = {"15.9", "15.10", "15.11"}  # какие хочешь удалить
# section_positions = [s for s in section_positions if s[1] not in to_remove_ids]


In [208]:
from collections import defaultdict

by_id = defaultdict(list)
for pos, sec_id, sec_name in section_positions:
    by_id[sec_id].append((pos, sec_name))

# выбираем последнюю (по позиции) для каждой секции
keep_pos = {max(v, key=lambda x: x[0])[0] for v in by_id.values()}

# оставляем только последние вхождения
section_positions = [s for s in section_positions if s[0] in keep_pos]

# сортируем по позиции
section_positions.sort(key=lambda x: x[0])

print(f"После удаления дублей осталось {len(section_positions)} секций.")


✅ После удаления дублей осталось 88 секций.


In [209]:
for i in section_positions:
    print(i)

(27871, '1.1', 'What Is Psychology')
(33261, '1.2', 'History of Psychology\nLEARNING OBJECTIVES\nBy the end of this section, you will be able to:')
(61296, '1.3', 'Contemporary Psychology\nLEARNING OBJECTIVES\nBy the end of this section, you will be able to:')
(84290, '1.4', 'Careers in Psychology\nLEARNING OBJECTIVES\nBy the end of this section, you will be able to:')
(107568, '2.1', 'Why Is Research Important')
(127030, '2.2', 'Approaches to Research\nLEARNING OBJECTIVES\nBy the end of this section, you will be able to:')
(151489, '2.3', 'Analyzing Findings\nLEARNING OBJECTIVES\nBy the end of this section, you will be able to:')
(188618, '2.4', 'Ethics\nLEARNING OBJECTIVES\nBy the end of this section, you will be able to:')
(219932, '3.1', 'Human Genetics\nLEARNING OBJECTIVES\nBy the end of this section, you will be able to:')
(241098, '3.2', 'Cells of the Nervous System\nLEARNING OBJECTIVES\nBy the end of this section, you will be able to:')
(258235, '3.3', 'Parts of the Nervous Sys

In [210]:
# text.lower().find(("12.2 Self-presentation\nlearning objectives").lower())
# print(text[1202006:1248718])

In [212]:
import re

cleaned_sections = []
for pos, sec_id, sec_name in section_positions:
    # Обрезаем всё после 'LEARNING OBJECTIVES' (включительно)
    cleaned_name = re.split(r'\n\s*LEARNING\s*OBJECTIVES', sec_name, flags=re.I)[0].strip()
    cleaned_sections.append((pos, sec_id, cleaned_name))

section_positions = cleaned_sections
section_positions

[(27871, '1.1', 'What Is Psychology'),
 (33261, '1.2', 'History of Psychology'),
 (61296, '1.3', 'Contemporary Psychology'),
 (84290, '1.4', 'Careers in Psychology'),
 (107568, '2.1', 'Why Is Research Important'),
 (127030, '2.2', 'Approaches to Research'),
 (151489, '2.3', 'Analyzing Findings'),
 (188618, '2.4', 'Ethics'),
 (219932, '3.1', 'Human Genetics'),
 (241098, '3.2', 'Cells of the Nervous System'),
 (258235, '3.3', 'Parts of the Nervous System'),
 (263568, '3.4', 'The Brain and Spinal Cord'),
 (292881, '3.5', 'The Endocrine System'),
 (320345, '4.1', 'What Is Consciousness'),
 (333971, '4.2', 'Sleep and Why We Sleep'),
 (341995, '4.3', 'Stages of Sleep'),
 (352911, '4.4', 'Sleep Problems and Disorders'),
 (371745, '4.5', 'Substance Use and Abuse'),
 (396331, '4.6', 'Other States of Consciousness'),
 (423238, '5.1', 'Sensation versus Perception'),
 (437207, '5.2', 'Waves and Wavelengths'),
 (444285, '5.3', 'Vision'),
 (464014, '5.4', 'Hearing'),
 (474291, '5.5', 'The Other Sens

In [219]:
toc = {
    "CHAPTER 1": {
        "title": "Introduction to Psychology",
        "sections": {
            "Introduction": 7,
            "1.1 What Is Psychology?": 8,
            "1.2 History of Psychology": 9,
            "1.3 Contemporary Psychology": 18,
            "1.4 Careers in Psychology": 26,
        }
    },
    "CHAPTER 2": {
        "title": "Psychological Research",
        "sections": {
            "Introduction": 35,
            "2.1 Why Is Research Important?": 36,
            "2.2 Approaches to Research": 41,
            "2.3 Analyzing Findings": 48,
            "2.4 Ethics": 59,
        }
    },
    "CHAPTER 3": {
        "title": "Biopsychology",
        "sections": {
            "Introduction": 71,
            "3.1 Human Genetics": 72,
            "3.2 Cells of the Nervous System": 78,
            "3.3 Parts of the Nervous System": 84,
            "3.4 The Brain and Spinal Cord": 86,
            "3.5 The Endocrine System": 97,
        }
    },
    "CHAPTER 4": {
        "title": "States of Consciousness",
        "sections": {
            "Introduction": 109,
            "4.1 What Is Consciousness?": 110,
            "4.2 Sleep and Why We Sleep": 114,
            "4.3 Stages of Sleep": 117,
            "4.4 Sleep Problems and Disorders": 121,
            "4.5 Substance Use and Abuse": 126,
            "4.6 Other States of Consciousness": 134,
        }
    },
    "CHAPTER 5": {
        "title": "Sensation and Perception",
        "sections": {
            "Introduction": 145,
            "5.1 Sensation versus Perception": 146,
            "5.2 Waves and Wavelengths": 149,
            "5.3 Vision": 153,
            "5.4 Hearing": 161,
            "5.5 The Other Senses": 164,
            "5.6 Gestalt Principles of Perception": 168,
        }
    },
    "CHAPTER 6": {
        "title": "Learning",
        "sections": {
            "Introduction": 181,
            "6.1 What Is Learning?": 182,
            "6.2 Classical Conditioning": 183,
            "6.3 Operant Conditioning": 192,
            "6.4 Observational Learning (Modeling)": 203,
        }
    },
    "CHAPTER 7": {
        "title": "Thinking and Intelligence",
        "sections": {
            "Introduction": 213,
            "7.1 What Is Cognition?": 214,
            "7.2 Language": 218,
            "7.3 Problem Solving": 222,
            "7.4 What Are Intelligence and Creativity?": 228,
            "7.5 Measures of Intelligence": 231,
            "7.6 The Source of Intelligence": 237,
        }
    },
    "CHAPTER 8": {
        "title": "Memory",
        "sections": {
            "Introduction": 247,
            "8.1 How Memory Functions": 248,
            "8.2 Parts of the Brain Involved with Memory": 255,
            "8.3 Problems with Memory": 259,
            "8.4 Ways to Enhance Memory": 269,
        }
    },
    "CHAPTER 9": {
        "title": "Lifespan Development",
        "sections": {
            "Introduction": 279,
            "9.1 What Is Lifespan Development?": 280,
            "9.2 Lifespan Theories": 284,
            "9.3 Stages of Development": 292,
            "9.4 Death and Dying": 313,
        }
    },
    "CHAPTER 10": {
        "title": "Emotion and Motivation",
        "sections": {
            "Introduction": 321,
            "10.1 Motivation": 322,
            "10.2 Hunger and Eating": 328,
            "10.3 Sexual Behavior": 334,
            "10.4 Emotion": 342,
        }
    },
    "CHAPTER 11": {
        "title": "Personality",
        "sections": {
            "Introduction": 359,
            "11.1 What Is Personality?": 360,
            "11.2 Freud and the Psychodynamic Perspective": 362,
            "11.3 Neo-Freudians: Adler, Erikson, Jung, and Horney": 368,
            "11.4 Learning Approaches": 373,
            "11.5 Humanistic Approaches": 377,
            "11.6 Biological Approaches": 378,
            "11.7 Trait Theorists": 379,
            "11.8 Cultural Understandings of Personality": 384,
            "11.9 Personality Assessment": 386,
        }
    },
    "CHAPTER 12": {
        "title": "Social Psychology",
        "sections": {
            "Introduction": 399,
            "12.1 What Is Social Psychology?": 400,
            "12.2 Self-presentation": 406,
            "12.3 Attitudes and Persuasion": 409,
            "12.4 Conformity, Compliance, and Obedience": 415,
            "12.5 Prejudice and Discrimination": 422,
            "12.6 Aggression": 429,
            "12.7 Prosocial Behavior": 432,
        }
    },
    "CHAPTER 13": {
        "title": "Industrial-Organizational Psychology",
        "sections": {
            "Introduction": 447,
            "13.1 What Is Industrial and Organizational Psychology?": 448,
            "13.2 Industrial Psychology: Selecting and Evaluating Employees": 456,
            "13.3 Organizational Psychology: The Social Dimension of Work": 467,
            "13.4 Human Factors Psychology and Workplace Design": 477,
        }
    },
    "CHAPTER 14": {
        "title": "Stress, Lifestyle, and Health",
        "sections": {
            "Introduction": 485,
            "14.1 What Is Stress?": 486,
            "14.2 Stressors": 496,
            "14.3 Stress and Illness": 502,
            "14.4 Regulation of Stress": 514,
            "14.5 The Pursuit of Happiness": 521,
        }
    },
    "CHAPTER 15": {
        "title": "Psychological Disorders",
        "sections": {
            "Introduction": 537,
            "15.1 What Are Psychological Disorders?": 538,
            "15.2 Diagnosing and Classifying Psychological Disorders": 542,
            "15.3 Perspectives on Psychological Disorders": 545,
            "15.4 Anxiety Disorders": 548,
            "15.5 Obsessive-Compulsive and Related Disorders": 554,
            "15.6 Posttraumatic Stress Disorder": 558,
            "15.7 Mood and Related Disorders": 560,
            "15.8 Schizophrenia": 570,
            "15.9 Dissociative Disorders": 574,
            "15.10 Disorders in Childhood": 576,
            "15.11 Personality Disorders": 582,
        }
    },
    "CHAPTER 16": {
        "title": "Therapy and Treatment",
        "sections": {
            "Introduction": 599,
            "16.1 Mental Health Treatment: Past and Present": 600,
            "16.2 Types of Treatment": 605,
            "16.3 Treatment Modalities": 617,
            "16.4 Substance-Related and Addictive Disorders: A Special Case": 621,
            "16.5 The Sociocultural Model and Therapy Utilization": 623,
        }
    },
}


In [228]:
# Сопоставим секцию с названием главы
chapter_map = {}
for ch_key, ch_data in toc.items():
    chapter_num = ch_key.split()[-1]  # например, 'CHAPTER 7' → '7'
    chapter_title = ch_data["title"]
    for sec_title in ch_data["sections"].keys():
        # В sec_title содержится строка вроде '7.3 Problem Solving'
        match = re.match(r'(\d+)\.(\d+)\s+(.*)', sec_title)
        if match:
            sec_num = match.group(1) + "." + match.group(2)
            chapter_map[sec_num] = chapter_title


{'1.1': 'Introduction to Psychology', '1.2': 'Introduction to Psychology', '1.3': 'Introduction to Psychology', '1.4': 'Introduction to Psychology', '2.1': 'Psychological Research', '2.2': 'Psychological Research', '2.3': 'Psychological Research', '2.4': 'Psychological Research', '3.1': 'Biopsychology', '3.2': 'Biopsychology', '3.3': 'Biopsychology', '3.4': 'Biopsychology', '3.5': 'Biopsychology', '4.1': 'States of Consciousness', '4.2': 'States of Consciousness', '4.3': 'States of Consciousness', '4.4': 'States of Consciousness', '4.5': 'States of Consciousness', '4.6': 'States of Consciousness', '5.1': 'Sensation and Perception', '5.2': 'Sensation and Perception', '5.3': 'Sensation and Perception', '5.4': 'Sensation and Perception', '5.5': 'Sensation and Perception', '5.6': 'Sensation and Perception', '6.1': 'Learning', '6.2': 'Learning', '6.3': 'Learning', '6.4': 'Learning', '7.1': 'Thinking and Intelligence', '7.2': 'Thinking and Intelligence', '7.3': 'Thinking and Intelligence', '

In [231]:
def slugify(text: str) -> str:
    """Преобразует текст в формат для ссылки: строчные, подчёркивания, без знаков препинания."""
    text = text.lower()
    text = re.sub(r'[^a-z0-9\s]', '', text)  # убираем пунктуацию
    text = re.sub(r'\s+', '_', text.strip())  # заменяем пробелы на _
    return text

def find_section_for_chunk(start_index, section_positions):
    """Возвращает секцию в формате 'chapter_title/section_title'."""
    current_section = "unknown_section"
    for i, (pos, sec_id, sec_name) in enumerate(section_positions):
        if start_index < pos:
            if i == 0:
                return current_section
            prev = section_positions[i - 1]
            sec_id, sec_name = prev[1], prev[2]

            # 🔧 Попытка получить название главы
            chapter_title = None
            if sec_id in chapter_map:
                chapter_title = chapter_map[sec_id]
            else:
                # пробуем взять номер главы (например, "3")
                chapter_title = chapter_map.get(sec_id.split('.')[0], None)

            if not chapter_title:
                chapter_title = "unknown_chapter"

            # Делаем slug (нормализованные имена)
            chap_slug = slugify(chapter_title)
            sec_slug = slugify(sec_name)
            return f"{chap_slug}/{sec_slug}"

    # если чанк после последней секции
    last = section_positions[-1]
    sec_id, sec_name = last[1], last[2]

    chapter_title = None
    if sec_id in chapter_map:
        chapter_title = chapter_map[sec_id]
    else:
        chapter_title = chapter_map.get(sec_id.split('.')[0], None)

    if not chapter_title:
        chapter_title = "unknown_chapter"

    chap_slug = slugify(chapter_title)
    sec_slug = slugify(sec_name)
    return f"{chap_slug}/{sec_slug}"


In [214]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1500, chunk_overlap=100, separators=["\n\n", "\n", " ", ""])

In [232]:
chunks = []
last_pos = 0
for chunk in text_splitter.split_text(text):
    if not chunk.strip():  # пропускаем пустые чанки
        continue

    start_index = text.find(chunk, last_pos)
    if start_index == -1:
        # если find не нашёл (например, повтор текста), пробуем искать с начала
        start_index = text.find(chunk)
        if start_index == -1:
            # если вообще не нашли — пропускаем этот чанк
            print(f"Не найден чанк{chunk} в тексте, пропущен.")
            continue

    page = find_page_for_chunk(start_index, page_start_positions)
    section = find_section_for_chunk(start_index, section_positions)
    chunks.append({"text": chunk, "page": page, "section": section})

    last_pos = start_index + len(chunk)

print(f"Получено {len(chunks)} чанков.")

Получено 1930 чанков.


In [233]:
vectors = embedding_model.encode([chunk["text"] for chunk in chunks],
                                 batch_size=32, device=device, normalize_embeddings=True, show_progress_bar=True).tolist()

Batches:   0%|          | 0/61 [00:00<?, ?it/s]

In [234]:
batch_size = 64

for i in tqdm(range(0, len(vectors), batch_size)):
    batch_points = [
        models.PointStruct(
            id=str(uuid.uuid4()),
            vector=vectors[j],
            payload={
                'text': chunks[j]["text"],
                'page': chunks[j]["page"],
                'section': chunks[j]["section"],
            }

        )
        for j in range(i, min(i + batch_size, len(vectors)))
    ]
    client.upsert(collection_name='psycology_e5', points=batch_points)


  0%|          | 0/31 [00:00<?, ?it/s]

# Submission with sections

In [235]:
def semantic_search(client, query, limit=10, collection_name="psycology_e5"):
    """
    Выполняет семантический поиск в коллекции Qdrant.

    Аргументы:
        client: экземпляр QdrantClient
        query: текстовый запрос пользователя
        limit: количество возвращаемых чанков (по умолчанию 10)
        collection_name: имя коллекции (по умолчанию 'psycology_e5')

    Возвращает:
        Список словарей формата:
        [
            {
                "text": "...",          # сам чанк
                "page": 123,            # страница, если есть в payload
                "score": 0.87           # косинусная близость
            },
            ...
        ]
    """
    # Кодируем запрос
    query_vector = embedding_model.encode(
        query,
        normalize_embeddings=True,
        device=device
    ).tolist()

    # Делаем запрос в Qdrant
    hits = client.search(
        collection_name=collection_name,
        query_vector=query_vector,
        limit=limit
    )

    # Безопасно обрабатываем результаты
    if not hits:
        print("Предупреждение: по запросу ничего не найдено.")
        return []

    # Собираем полезную информацию из результатов
    results = []
    for hit in hits:
        payload = hit.payload or {}
        results.append({
            "text": payload.get("text", ""),
            "page": payload.get("page", None),
            "section": payload.get("section", None),
            "score": hit.score
        })

    return results


In [236]:
def llm_answer(query, context):
    """
    Генерирует ответ на вопрос по фрагментам учебника (RAG).
    Использует Qwen2-1.5B-Instruct для точного, достоверного ответа.
    """
    prompt = f"""Текст из учебника психологии:
{context}

Вопрос:
{query}"""

    messages = [
        {
            "role": "system",
            "content": (
                "You are a very skeptical scientist in the field of psychology. You will receive a context consisting of text clippings from a book on the desired topic. Your task is to answer the user as accurately and honestly as possible. Make sure that the answer is detailed, specific, and directly related to the question. Do not add information that is not directly supported by the provided clippings from the book. If there is no direct answer in the text, tell me about it honestly."
            ),
        },
        {"role": "user", "content": prompt},
    ]

    output = generation_pipeline(
        messages,
        max_new_tokens=220,
        do_sample=True,
        temperature=0.1,
        top_p=0.9,
    )

    # Проверяем структуру вывода, т.к. она может отличаться между версиями Transformers
    if isinstance(output[0]["generated_text"], list):
        # новый формат: список сообщений
        return output[0]["generated_text"][-1]["content"]
    elif isinstance(output[0]["generated_text"], str):
        # старый формат: просто строка
        return output[0]["generated_text"]
    else:
        # fallback
        return str(output[0])


In [237]:
import json
import pandas as pd

queries = json.load(open("/kaggle/input/casml-dataset/Dataset_RAG (1)/queries.json"))

results = []
for q in tqdm(queries):
    query = q["question"]
    query_id = q["query_id"]

    relevant_chunks = semantic_search(client, query, limit=5)
    context = "\n\n".join([chunk["text"] for chunk in relevant_chunks])
    pages = sorted({chunk["page"] for chunk in relevant_chunks if chunk.get("page") is not None})
    sections = sorted({chunk["section"] for chunk in relevant_chunks if chunk.get("section") not in [None, "unknown_section"]})
    references = json.dumps({"sections": sections, "pages": pages})

    answer = llm_answer(query, context)

    results.append({
        "ID": query_id,
        "context": context,
        "answer": answer,
        "references": references
    })


  0%|          | 0/50 [00:00<?, ?it/s]

/tmp/ipykernel_48/549817120.py:30: DeprecationWarning: `search` method is deprecated and will be removed in the future. Use `query_points` instead.
  hits = client.search(


In [238]:
df = pd.DataFrame(results)
df.to_csv("submission_basic_rag_sections3.csv", index=False)
print("submission_basic_rag_dections3.csv saved")

submission_basic_rag_dections3.csv saved


In [239]:
results[1]

{'ID': '2',
 'context': 'the other hand, serve as interconnected information processors that are essential for all of the tasks of the\nnervous system. This section briefly describes the structure and function of neurons.\nNeuron Structure\nNeurons are the central building blocks of the nervous system, 100 billion strong at birth. Like all cells,\nneurons consist of several different parts, each serving a specialized function (Figure 3.8). A neuron’s outer\nsurface is made up of a semipermeable membrane. This membrane allows smaller molecules and molecules\nwithout an electrical charge to pass through it, while stopping larger or highly charged molecules.\nFIGURE 3.8 This illustration shows a prototypical neuron, which is being myelinated by a glial cell.\nThe nucleus of the neuron is located in the soma, or cell body. The soma has branching extensions known as\ndendrites. The neuron is a small information processor, and dendrites serve as input sites where signals are\n\nthe other han